# 09_B_TEST_TUNING — Hyperparameter Tuning Eksperimen LSTM (v2 — F2 Drowsy)

##  PERUBAHAN VERSI v2 (28 April 2026)

###  Bug Fixes
1. **Early stopping → F2-Score Drowsy** (sebelumnya F1 Macro)
   - Konsekuensi: model dipilih berdasarkan recall drowsy yang dipenalti precision lemah,
     selaras dengan kebutuhan sistem safety-critical.
2. **Scheduler step fix** — hapus noise `va_f1/100` di `scheduler.step()`.
3. **Per-class metrics di setiap epoch** — `recall_drowsy`, `precision_drowsy`,
   `f2_drowsy` divisualisasikan tiap epoch.

###  Eksperimen Baru (5 grup)
- **EXP_G**: CE + Label Smoothing = 0.0 (uji apakah LS=0.05 menyebabkan underconfidence)
- **EXP_H**: WD=5e-5 + LR=8e-5 (sangat kecil, cek over-regularization)
- **EXP_I**: Focal γ=1.5 (softer focal, kompromi CE vs γ=2)
- **EXP_J** (SWIN only): Focal γ=2 × Aug × WD=1e-4 (stack untuk minimisasi FN)
- **EXP_K** (SWIN only): max_epochs=120, patience=30, T_0=20 (runway panjang)

###  Metrik Utama
- **F2-Score Drowsy** (β=2, recall di-prioritaskan 4× dari precision)
  Formula: F2 = (1+4) × P × R / (4P + R) → recall punya bobot 4× precision.
- **Recall Drowsy** sebagai constraint utama (target ≥ 0.80).
- **F1 Macro** tetap dilaporkan sebagai sanity check.
- **Drowsy Missed (FN)** sebagai metrik safety (target ≤ 110, 50% dari baseline 204).

---

##  Konteks Skripsi
**Judul**: Deteksi Kantuk Pengemudi Real-time Swin Transformer + LSTM
**Dataset**: NTHU-DDD2 (drowsy=42.1%, notdrowsy=57.9% di test set, total 1302 sampel)
**Kelas**: 0=drowsy, 1=notdrowsy
**Hardware**: RTX 3060 12GB | Ryzen 5 5600 | 16GB RAM
**Target Akhir**: F2 Drowsy ≥ 0.75 | Recall Drowsy ≥ 0.80 | F1 Macro ≥ 0.70


#  Import & Konfigurasi Global

In [27]:
# ============================================================
# CELL 1 — Import & Konfigurasi Global (v2)
# ============================================================
# PERUBAHAN v2:
#   + Import fbeta_score untuk F2-Score Drowsy (β=2)
# ============================================================

import os, gc, json, time, random, warnings
from copy import deepcopy
from itertools import product

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import (
    CosineAnnealingWarmRestarts,
    OneCycleLR,
    ReduceLROnPlateau,
)
from sklearn.metrics import (
    f1_score, fbeta_score, accuracy_score, precision_score,
    recall_score, confusion_matrix, roc_auc_score,
)
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ── SEED ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True   # input fixed → speedup

# ── DEVICE ────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── PATH — Sesuaikan dengan path PC kamu ─────────────────────
BASE_DIR  = r"C:\kuliah-sementara\SKRIPSI"
SEQ_DIR   = os.path.join(BASE_DIR, "sequences_nthuddd2")
SAVE_DIR  = os.path.join(BASE_DIR, "tuning_results_nthuddd2")
os.makedirs(SAVE_DIR, exist_ok=True)

# ── DATA CONFIG (TIDAK BERUBAH) ───────────────────────────────
MODELS      = ["CNN_BASIC", "MOBILENET", "VGG19", "SWIN"]
SPLITS      = ["train", "val", "test"]
INPUT_DIM   = 512
WINDOW_SIZE = 30
LABEL_MAP   = {0: "drowsy", 1: "notdrowsy"}
NUM_CLASSES = 2
NUM_WORKERS = 0     # Windows: wajib 0
DROWSY_CLASS_IDX   = 0  # ← KELAS POSITIF UNTUK F2-SCORE
NOTDROWSY_CLASS_IDX = 1

# ── PRINT KONFIGURASI ─────────────────────────────────────────
print("=" * 65)
print("  09_B_TEST_TUNING v2 — F2-Score Drowsy Optimization")
print("=" * 65)
print(f"  Device      : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    total_vram = props.total_memory / 1e9
    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM Total  : {total_vram:.1f} GB")
    print(f"  cuDNN bench : {torch.backends.cudnn.benchmark} (enabled for speed)")
print(f"  SEQ_DIR     : {SEQ_DIR}")
print(f"  SAVE_DIR    : {SAVE_DIR}")
print(f"  Input Dim   : {INPUT_DIM} | Window : {WINDOW_SIZE} frames")
print(f"  Drowsy idx  : {DROWSY_CLASS_IDX} (kelas positif untuk F2)")
print("=" * 65)
print("✓ CELL 1 selesai.")


  09_B_TEST_TUNING v2 — F2-Score Drowsy Optimization
  Device      : cuda
  GPU         : NVIDIA GeForce RTX 3060
  VRAM Total  : 12.9 GB
  cuDNN bench : True (enabled for speed)
  SEQ_DIR     : C:\kuliah-sementara\SKRIPSI\sequences_nthuddd2
  SAVE_DIR    : C:\kuliah-sementara\SKRIPSI\tuning_results_nthuddd2
  Input Dim   : 512 | Window : 30 frames
  Drowsy idx  : 0 (kelas positif untuk F2)
✓ CELL 1 selesai.


# Validasi File Sekuens

In [28]:
# ============================================================
# CELL 2 — Validasi File Sekuens Input
# ============================================================
# Pastikan semua 12 file .pt dari notebook 08 tersedia.
# Sama persis dengan file 09 asli.
# ============================================================

print("[VALIDASI] Mengecek file sekuens input...\n")

all_ok = True
for model in MODELS:
    for split in SPLITS:
        fname = f"{model}_Seq_{split.capitalize()}_NTHUD.pt"
        fpath = os.path.join(SEQ_DIR, model, fname)
        exists = os.path.exists(fpath)
        if exists:
            data    = torch.load(fpath, map_location="cpu", weights_only=False)
            n_seq   = data["features"].shape[0]
            n_drow  = int((data["labels"] == 0).sum())
            n_notd  = int((data["labels"] == 1).sum())
            balance = n_drow / n_seq * 100
            print(f"  [OK] {fname}")
            print(f"       Shape={list(data['features'].shape)} | "
                  f"drowsy={n_drow:,} ({balance:.1f}%) | notdrowsy={n_notd:,}")
        else:
            print(f"  [MISSING] {fname}")
            all_ok = False

print()
if not all_ok:
    raise FileNotFoundError(
        "Beberapa file .pt tidak ditemukan!\n"
        "Pastikan notebook 08 sudah dijalankan."
    )
print("✓ Semua file sekuens tersedia. Siap eksperimen tuning.")

[VALIDASI] Mengecek file sekuens input...

  [OK] CNN_BASIC_Seq_Train_NTHUD.pt
       Shape=[8053, 30, 512] | drowsy=4,502 (55.9%) | notdrowsy=3,551
  [OK] CNN_BASIC_Seq_Val_NTHUD.pt
       Shape=[3634, 30, 512] | drowsy=2,100 (57.8%) | notdrowsy=1,534
  [OK] CNN_BASIC_Seq_Test_NTHUD.pt
       Shape=[1302, 30, 512] | drowsy=548 (42.1%) | notdrowsy=754
  [OK] MOBILENET_Seq_Train_NTHUD.pt
       Shape=[8053, 30, 512] | drowsy=4,502 (55.9%) | notdrowsy=3,551
  [OK] MOBILENET_Seq_Val_NTHUD.pt
       Shape=[3634, 30, 512] | drowsy=2,100 (57.8%) | notdrowsy=1,534
  [OK] MOBILENET_Seq_Test_NTHUD.pt
       Shape=[1302, 30, 512] | drowsy=548 (42.1%) | notdrowsy=754
  [OK] VGG19_Seq_Train_NTHUD.pt
       Shape=[8053, 30, 512] | drowsy=4,502 (55.9%) | notdrowsy=3,551
  [OK] VGG19_Seq_Val_NTHUD.pt
       Shape=[3634, 30, 512] | drowsy=2,100 (57.8%) | notdrowsy=1,534
  [OK] VGG19_Seq_Test_NTHUD.pt
       Shape=[1302, 30, 512] | drowsy=548 (42.1%) | notdrowsy=754
  [OK] SWIN_Seq_Train_NTHUD.pt
     

# Dataset + DataLoader + Temporal Augmentation

In [29]:
# ============================================================
# CELL 3 — Dataset Class + DataLoader
# ============================================================
# PERUBAHAN vs File 09:
#   + SequenceDatasetAugmented: mendukung temporal augmentation
#     - Frame Dropping (1-2 frame, copy neighbor)
#     - Gaussian Noise kecil (std=0.01)
#   Augmentasi HANYA aktif pada training set (augment=True).
#   Val/Test tetap bersih tanpa augmentasi.
# ============================================================

class SequenceDatasetAugmented(Dataset):
    """
    Dataset wrapper dengan optional Temporal Augmentation untuk training.

    Temporal Augmentation:
    1. Frame Dropping  : Hapus 1-2 frame secara acak dari 30 frame,
                         gantikan dengan copy frame tetangga terdekat.
                         Tujuan: simulasi kamera lag / dropped frame.
    2. Gaussian Noise  : Tambahkan noise N(0, 0.01) ke semua fitur.
                         Tujuan: membuat LSTM lebih robust terhadap
                         variasi kecil pada fitur CNN.

    Args:
        features    : Tensor [N, T, D]
        labels      : Tensor [N]
        mean / std  : Statistik dari training set (Z-Score normalisasi)
        augment     : True = aktifkan augmentasi (hanya untuk train)
        aug_prob_fd : Probabilitas frame dropping per sampel
        aug_prob_gn : Probabilitas gaussian noise per sampel
        n_drop_max  : Maksimum frame yang di-drop (1 atau 2)
        noise_std   : Standar deviasi Gaussian noise
    """
    def __init__(
        self,
        features    : torch.Tensor,
        labels      : torch.Tensor,
        mean        : torch.Tensor = None,
        std         : torch.Tensor = None,
        augment     : bool  = False,
        aug_prob_fd : float = 0.3,   # 30% chance frame dropping
        aug_prob_gn : float = 0.3,   # 30% chance gaussian noise
        n_drop_max  : int   = 1,     # Max 1 frame di-drop (konservatif)
        noise_std   : float = 0.01,  # Noise sangat kecil
    ):
        self.features    = features.float()
        self.labels      = labels.long()
        self.augment     = augment
        self.aug_prob_fd = aug_prob_fd
        self.aug_prob_gn = aug_prob_gn
        self.n_drop_max  = n_drop_max
        self.noise_std   = noise_std

        # Z-Score Normalisasi (dari train stats)
        if mean is not None and std is not None:
            std_safe = std.clamp(min=1e-8)
            self.features = (self.features - mean) / std_safe

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feat = self.features[idx].clone()  # [T=30, D=512]

        if self.augment:
            # ── Augmentasi 1: Frame Dropping ──────────────────
            # Hapus 1–n_drop_max frame secara acak, gantikan
            # dengan copy frame tetangga (copy-paste nearest)
            if random.random() < self.aug_prob_fd:
                n_drop = random.randint(1, self.n_drop_max)
                T = feat.shape[0]
                drop_idx = sorted(
                    random.sample(range(T), n_drop), reverse=True
                )
                for di in drop_idx:
                    # Pilih tetangga: sebelum atau sesudah
                    neighbor = di - 1 if di > 0 else di + 1
                    neighbor = max(0, min(neighbor, T - 1))
                    feat[di] = feat[neighbor].clone()

            # ── Augmentasi 2: Gaussian Noise ──────────────────
            # Noise kecil untuk mencegah LSTM "menghafal" pola
            if random.random() < self.aug_prob_gn:
                noise = torch.randn_like(feat) * self.noise_std
                feat = feat + noise

        return feat, self.labels[idx]


def load_split_data(model_name: str, split: str):
    """Load file .pt untuk satu backbone + satu split."""
    fname = f"{model_name}_Seq_{split.capitalize()}_NTHUD.pt"
    fpath = os.path.join(SEQ_DIR, model_name, fname)
    data  = torch.load(fpath, map_location="cpu", weights_only=False)
    return data["features"].float(), data["labels"].long()


def compute_train_stats(features: torch.Tensor):
    """Hitung mean & std dari training set untuk Z-Score normalisasi."""
    N, T, D = features.shape
    flat    = features.reshape(-1, D)
    return flat.mean(dim=0), flat.std(dim=0)


def build_dataloaders(
    model_name   : str,
    batch_size   : int  = 64,
    use_aug      : bool = False,
    aug_prob_fd  : float = 0.3,
    aug_prob_gn  : float = 0.3,
    n_drop_max   : int  = 1,
    noise_std    : float = 0.01,
):
    """
    Build train/val/test DataLoader untuk satu backbone.
    Normalisasi dari train set, augmentasi hanya pada train jika use_aug=True.

    Returns:
        loaders      : dict {train, val, test}
        class_weight : Tensor untuk weighted loss
        norm_stats   : dict {mean, std}
    """
    # Load semua split
    train_feat, train_lab = load_split_data(model_name, "train")
    val_feat,   val_lab   = load_split_data(model_name, "val")
    test_feat,  test_lab  = load_split_data(model_name, "test")

    # Normalisasi dari train saja (cegah data leakage)
    mean, std = compute_train_stats(train_feat)

    # Dataset
    train_ds = SequenceDatasetAugmented(
        train_feat, train_lab, mean, std,
        augment=use_aug,
        aug_prob_fd=aug_prob_fd,
        aug_prob_gn=aug_prob_gn,
        n_drop_max=n_drop_max,
        noise_std=noise_std,
    )
    val_ds  = SequenceDatasetAugmented(val_feat,  val_lab,  mean, std, augment=False)
    test_ds = SequenceDatasetAugmented(test_feat, test_lab, mean, std, augment=False)

    # Class Weight: w_c = N_total / (num_classes × N_c)
    label_counts = Counter(train_lab.numpy())
    n_total      = len(train_lab)
    weights      = torch.tensor([
        n_total / (NUM_CLASSES * label_counts[c])
        for c in range(NUM_CLASSES)
    ], dtype=torch.float32)

    pin = (DEVICE.type == "cuda")
    loaders = {
        "train": DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=pin),
        "val"  : DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin),
        "test" : DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin),
    }
    norm_stats = {"mean": mean, "std": std}

    print(f"  [DataLoader] {model_name} | batch={batch_size} | aug={'ON' if use_aug else 'OFF'}")
    print(f"  [Norm]       mean={mean.mean():.4f}  std={std.mean():.4f}")
    print(f"  [ClassW]     drowsy={weights[0]:.4f}  notdrowsy={weights[1]:.4f}")

    return loaders, weights, norm_stats


print("✓ CELL 3 — Dataset + DataLoader + Augmentasi terdefinisi.")

✓ CELL 3 — Dataset + DataLoader + Augmentasi terdefinisi.


# Arsitektur Model (Fleksibel + GELU)

In [30]:
# ============================================================
# CELL 4 — Arsitektur Model
# ============================================================
# PERUBAHAN vs File 09:
#   + Aktivasi FC bisa dipilih: ReLU / GELU / SiLU
#     - GELU direkomendasikan untuk Swin (konsisten dengan
#       aktivasi internal Swin Transformer)
#   + LSTM hidden_dim bisa 256 atau 512 (tunable)
#   + num_layers bisa 2 atau 3 (tunable)
#   + Semua parameter dipass lewat config dict (bukan global)
# ============================================================

class AdditiveAttention(nn.Module):
    """
    Additive (Bahdanau-style) Attention untuk output LSTM.
    Input  : [B, T, H]
    Output : context [B, H], weights [B, T]
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out: torch.Tensor):
        energy  = torch.tanh(self.attn(lstm_out))       # [B, T, H]
        scores  = self.v(energy).squeeze(-1)             # [B, T]
        weights = torch.softmax(scores, dim=-1)          # [B, T]
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)  # [B, H]
        return context, weights


# ── Activation Factory ────────────────────────────────────────
_ACTIVATION_MAP = {
    "relu" : nn.ReLU,
    "gelu" : nn.GELU,
    "silu" : nn.SiLU,
    "elu"  : nn.ELU,
}

def get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name not in _ACTIVATION_MAP:
        raise ValueError(f"Aktivasi '{name}' tidak dikenal. Pilih: {list(_ACTIVATION_MAP)}")
    return _ACTIVATION_MAP[name]()


class DrowsinessLSTM(nn.Module):
    """
    Arsitektur utama: Stacked LSTM/BiLSTM + Attention + FC Classifier.

    Perbedaan vs File 09:
    - Aktivasi FC bisa dipilih via `fc_activation` ('relu'/'gelu'/'silu')
    - hidden_dim, num_layers dapat diatur dari config eksperimen
    """
    def __init__(
        self,
        input_dim     : int   = 512,
        hidden_dim    : int   = 256,
        num_layers    : int   = 2,
        num_classes   : int   = 2,
        bidirectional : bool  = False,
        use_attention : bool  = True,
        lstm_dropout  : float = 0.3,
        fc_dropout    : float = 0.5,
        fc_activation : str   = "relu",   # ← BARU: pilihan aktivasi
    ):
        super().__init__()
        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.hidden_dim    = hidden_dim
        self.num_layers    = num_layers
        num_dir            = 2 if bidirectional else 1
        lstm_out_dim       = hidden_dim * num_dir

        # ── Layer Normalisasi Input ──────────────────────────
        self.input_norm = nn.LayerNorm(input_dim)

        # ── LSTM / BiLSTM ────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = lstm_dropout if num_layers > 1 else 0.0,
        )

        # ── Attention ────────────────────────────────────────
        if use_attention:
            self.attention = AdditiveAttention(lstm_out_dim)
        else:
            self.attention = None
        classifier_in = lstm_out_dim

        # ── FC Classifier (dengan pilihan aktivasi) ──────────
        act = get_activation(fc_activation)
        self.classifier = nn.Sequential(
            nn.Dropout(fc_dropout),
            nn.Linear(classifier_in, 128),
            act,                            # ← GELU untuk Swin
            nn.Dropout(fc_dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor):
        x        = self.input_norm(x)                          # [B, T, D]
        lstm_out, (h_n, _) = self.lstm(x)                     # [B, T, H*dir]

        if self.use_attention:
            context, attn_w = self.attention(lstm_out)         # [B, H*dir]
        else:
            context = lstm_out[:, -1, :]
            attn_w  = None

        logits = self.classifier(context)                      # [B, C]
        return logits, attn_w


def build_model_from_cfg(cfg: dict) -> DrowsinessLSTM:
    """
    Bangun model dari dictionary konfigurasi eksperimen.
    Contoh cfg: {hidden_dim:256, num_layers:2, lstm_dropout:0.3,
                 fc_dropout:0.5, fc_activation:'gelu',
                 bidirectional:False, use_attention:True}
    """
    return DrowsinessLSTM(
        input_dim     = INPUT_DIM,
        hidden_dim    = cfg.get("hidden_dim",    256),
        num_layers    = cfg.get("num_layers",    2),
        num_classes   = NUM_CLASSES,
        bidirectional = cfg.get("bidirectional", False),
        use_attention = cfg.get("use_attention", True),
        lstm_dropout  = cfg.get("lstm_dropout",  0.3),
        fc_dropout    = cfg.get("fc_dropout",    0.5),
        fc_activation = cfg.get("fc_activation", "relu"),
    ).to(DEVICE)


# ── Sanity check arsitektur ───────────────────────────────────
print("[TEST] Memverifikasi semua kombinasi arsitektur...")
for act in ["relu", "gelu", "silu"]:
    for hidden in [256, 512]:
        cfg_test = dict(
            hidden_dim=hidden, num_layers=2, lstm_dropout=0.3,
            fc_dropout=0.5, fc_activation=act, bidirectional=False, use_attention=True
        )
        m   = build_model_from_cfg(cfg_test)
        x   = torch.randn(2, WINDOW_SIZE, INPUT_DIM).to(DEVICE)
        out, _ = m(x)
        assert out.shape == (2, NUM_CLASSES), f"Shape error: {out.shape}"
        del m, x, out
torch.cuda.empty_cache()
print("✓ Semua kombinasi arsitektur valid.")
print("✓ CELL 4 — Arsitektur Model terdefinisi.")

[TEST] Memverifikasi semua kombinasi arsitektur...
✓ Semua kombinasi arsitektur valid.
✓ CELL 4 — Arsitektur Model terdefinisi.


# Loss Functions (Focal Loss + Factory)

In [31]:
# ============================================================
# CELL 5 — Loss Functions
# ============================================================
# PERUBAHAN vs File 09:
#   + FocalLoss: down-weight easy examples, fokus ke hard ones
#     Referensi: Lin et al. (2017) "Focal Loss for Dense Object Detection"
#   + loss_factory(): pilih loss berdasarkan string config
#
# Kapan pakai Focal Loss?
# - Dataset ini: drowsy 42.1% vs notdrowsy 57.9% di test set
# - Imbalance moderat (bukan ekstrem), jadi Focal loss hanya
#   membantu jika dikombinasikan dengan class_weight
# - Untuk Swin: patut dicoba karena F1 drowsy masih rendah
# ============================================================

class FocalLoss(nn.Module):
    """
    Focal Loss = -α_t * (1 - p_t)^γ * log(p_t)

    Args:
        gamma        : Focusing parameter. 0 = CE biasa.
                       Saran: gamma=1.0 (moderat) atau gamma=2.0 (agresif)
        weight       : Class weight tensor (dari build_dataloaders)
        label_smoothing: Smoothing ε (0.0 - tidak aktif, 0.05 - ringan)
        reduction    : 'mean' (default)

    Note:
        Focal Loss + class_weight = kombinasi yang paling powerful
        untuk imbalanced binary classification di sistem safety-critical.
    """
    def __init__(
        self,
        gamma           : float = 2.0,
        weight          : torch.Tensor = None,
        label_smoothing : float = 0.0,
        reduction       : str   = "mean",
    ):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing
        self.reduction       = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # CrossEntropy dengan label smoothing (per-sample, reduction='none')
        ce_loss = F.cross_entropy(
            logits, targets,
            weight          = self.weight,
            label_smoothing = self.label_smoothing,
            reduction       = "none",
        )
        # p_t: probabilitas untuk kelas yang benar
        pt          = torch.exp(-ce_loss)
        focal_loss  = ((1.0 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


def build_loss(cfg: dict, class_weight: torch.Tensor) -> nn.Module:
    """
    Factory function untuk membangun loss dari konfigurasi eksperimen.

    cfg keys:
        loss_type       : 'ce' (CrossEntropy) atau 'focal'
        label_smoothing : float (0.0–0.1)
        focal_gamma     : float (1.0 atau 2.0, hanya untuk focal)
    """
    loss_type       = cfg.get("loss_type",        "ce")
    label_smoothing = cfg.get("label_smoothing",  0.0)
    focal_gamma     = cfg.get("focal_gamma",      2.0)
    cw              = class_weight.to(DEVICE)

    if loss_type == "focal":
        return FocalLoss(
            gamma           = focal_gamma,
            weight          = cw,
            label_smoothing = label_smoothing,
        ).to(DEVICE)
    else:  # "ce"
        return nn.CrossEntropyLoss(
            weight          = cw,
            label_smoothing = label_smoothing,
        ).to(DEVICE)


# ── Tes loss functions ────────────────────────────────────────
_dummy_logits  = torch.randn(4, 2).to(DEVICE)
_dummy_targets = torch.tensor([0, 1, 0, 1]).to(DEVICE)
_dummy_weight  = torch.tensor([0.89, 1.13]).to(DEVICE)

_fl = FocalLoss(gamma=2.0, weight=_dummy_weight)(_dummy_logits, _dummy_targets)
_ce = nn.CrossEntropyLoss(weight=_dummy_weight, label_smoothing=0.05)(
    _dummy_logits, _dummy_targets
)
print(f"  FocalLoss test  : {_fl.item():.4f}")
print(f"  CrossEntropy test: {_ce.item():.4f}")
del _dummy_logits, _dummy_targets, _dummy_weight, _fl, _ce
print("✓ CELL 5 — Loss Functions terdefinisi.")

  FocalLoss test  : 0.7103
  CrossEntropy test: 1.1207
✓ CELL 5 — Loss Functions terdefinisi.


# LR Range Test Utility

In [32]:
# ============================================================
# CELL 6 — LR Range Test Utility
# ============================================================
# Implementasi dari Smith (2017) arXiv:1803.09820
#
# Cara kerja:
# - LR dinaikkan secara eksponensial dari start_lr → end_lr
# - Catat loss di setiap step
# - Plot: pilih LR di area SEBELUM loss mulai naik
#         (biasanya titik kemiringan paling curam ke bawah)
#
# Hasil LR Range Test digunakan sebagai max_lr untuk:
# - CosineAnnealingWarmRestarts
# - OneCycleLR
# ============================================================

def lr_range_test(
    backbone_name : str,
    cfg           : dict,
    start_lr      : float = 1e-7,
    end_lr        : float = 1e-1,
    num_iter      : int   = 150,
    batch_size    : int   = 64,
    smooth_beta   : float = 0.9,    # Exponential smoothing untuk kurva
    save_plot     : bool  = True,
):
    """
    Jalankan LR Range Test untuk satu backbone + konfigurasi.

    Returns:
        best_lr : float — LR yang disarankan (di kemiringan terbesar)
        lrs     : list[float]
        losses  : list[float] (smoothed)
    """
    print(f"\n[LR Range Test] Backbone={backbone_name} | "
          f"LR {start_lr:.0e} → {end_lr:.0e} | {num_iter} iter")

    # Build data & model baru (fresh)
    loaders, class_weight, _ = build_dataloaders(
        backbone_name, batch_size=batch_size
    )
    model     = build_model_from_cfg(cfg)
    criterion = build_loss(cfg, class_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=start_lr,
        weight_decay=cfg.get("weight_decay", 1e-4)
    )

    # Multiplier LR per iterasi
    mult = (end_lr / start_lr) ** (1.0 / num_iter)
    lrs, raw_losses, smooth_losses = [], [], []
    avg_loss, best_loss = 0.0, float("inf")

    model.train()
    train_iter = iter(loaders["train"])

    for i in range(num_iter):
        try:
            feats, labels = next(train_iter)
        except StopIteration:
            train_iter = iter(loaders["train"])
            feats, labels = next(train_iter)

        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits, _ = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        # Exponential smoothing
        avg_loss = smooth_beta * avg_loss + (1 - smooth_beta) * loss.item()
        smoothed = avg_loss / (1 - smooth_beta ** (i + 1))  # Bias correction

        lrs.append(optimizer.param_groups[0]["lr"])
        raw_losses.append(loss.item())
        smooth_losses.append(smoothed)

        # Stop jika loss meledak (4× loss minimum)
        if smoothed < best_loss:
            best_loss = smoothed
        if smoothed > 4 * best_loss:
            print(f"  Loss diverged di iterasi {i+1}. Berhenti.")
            break

        # Naikkan LR
        for pg in optimizer.param_groups:
            pg["lr"] *= mult

    # ── Temukan LR optimal: kemiringan paling curam ke bawah ──
    # Hitung gradien kurva smoothed loss, pilih titik minimum
    if len(smooth_losses) > 10:
        grad      = np.gradient(smooth_losses)
        min_grad_idx = np.argmin(grad[5:]) + 5  # Skip 5 iterasi awal
        best_lr   = lrs[min_grad_idx] / 10.0   # Sedikit lebih konservatif
    else:
        best_lr = 1e-4

    print(f"  → Saran max_lr : {best_lr:.2e} "
          f"(LR di kemiringan terbesar = {lrs[min_grad_idx]:.2e})")

    # ── Plot ──────────────────────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(9, 4))
    ax.plot(lrs[:len(smooth_losses)], smooth_losses, lw=2, color="#2563eb", label="Loss (smoothed)")
    ax.axvline(best_lr, color="#dc2626", linestyle="--", lw=1.5,
               label=f"Saran max_lr = {best_lr:.2e}")
    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate (log scale)", fontsize=11)
    ax.set_ylabel("Loss", fontsize=11)
    ax.set_title(f"LR Range Test — {backbone_name}", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_plot:
        plot_dir = os.path.join(SAVE_DIR, "lr_range_tests")
        os.makedirs(plot_dir, exist_ok=True)
        plot_path = os.path.join(plot_dir, f"lr_range_{backbone_name}.png")
        plt.savefig(plot_path, dpi=120, bbox_inches="tight")
        print(f"  Plot disimpan: {plot_path}")
    plt.show()

    # Cleanup
    del model, loaders, optimizer, criterion
    gc.collect()
    torch.cuda.empty_cache()

    return best_lr, lrs, smooth_losses


# ── Contoh cara pakai (jalankan sebelum CELL 9 jika ingin LR test) ──
# Contoh: best_lr_swin, _, _ = lr_range_test("SWIN", cfg_swin_base)
# Lalu masukkan best_lr_swin ke EXPERIMENT_GRID di CELL 9

print("✓ CELL 6 — LR Range Test terdefinisi.")
print("  Cara pakai: lr_range_test('SWIN', cfg_dict)")
print("  Jalankan ini SEBELUM CELL 9 untuk mendapatkan LR optimal per backbone.")

✓ CELL 6 — LR Range Test terdefinisi.
  Cara pakai: lr_range_test('SWIN', cfg_dict)
  Jalankan ini SEBELUM CELL 9 untuk mendapatkan LR optimal per backbone.


# Training Core (CosineWarmRestart + Cyclical Momentum)


In [33]:
# ============================================================
# CELL 7 — Training Core Functions (v2 — F2 Drowsy)
# ============================================================
# PERUBAHAN v2:
#
#   1. evaluate() & train_epoch() return dict lengkap dengan:
#      - f1_macro, f2_drowsy, recall_drowsy, precision_drowsy
#      Backward-compatible: tuple unpacking lama tetap jalan via
#      _legacy_unpack().
#
#   2. Bug fix: scheduler.step(epoch + 1) — hapus noise va_f1/100.
#      Justifikasi: CosineAnnealingWarmRestarts butuh integer/fraction
#      epoch yang konsisten. Noise random merusak periode restart.
#
#   3. Tetap support OneCycleLR (per_batch step) dan plateau (val_f1).
# ============================================================

def _compute_metrics(preds, labels):
    """
    Hitung dict metrik lengkap:
      acc, f1_macro, f2_drowsy, recall_drowsy, precision_drowsy.
    F2-Score: β=2 → recall di-bobot 4× precision (formula: F2 = 5PR/(4P+R)).
    """
    preds  = np.asarray(preds)
    labels = np.asarray(labels)
    return {
        "acc"              : accuracy_score(labels, preds),
        "f1_macro"         : f1_score(labels, preds, average="macro", zero_division=0),
        # F2-Score binary, kelas positif = DROWSY (0)
        "f2_drowsy"        : fbeta_score(
            labels, preds, beta=2.0, pos_label=DROWSY_CLASS_IDX,
            average="binary", zero_division=0
        ),
        "recall_drowsy"    : recall_score(
            labels, preds, pos_label=DROWSY_CLASS_IDX,
            average="binary", zero_division=0
        ),
        "precision_drowsy" : precision_score(
            labels, preds, pos_label=DROWSY_CLASS_IDX,
            average="binary", zero_division=0
        ),
    }


def train_epoch(model, loader, criterion, optimizer, scheduler=None,
                sched_type="per_epoch", max_grad_norm=1.0):
    """
    Satu epoch training.
    Returns: dict {loss, acc, f1_macro, f2_drowsy, recall_drowsy, precision_drowsy}
    """
    model.train()
    total_loss            = 0.0
    all_preds, all_labels = [], []

    for feats, labels in loader:
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits, _ = model(feats)
        loss      = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()

        if sched_type == "per_batch" and scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    metrics  = _compute_metrics(all_preds, all_labels)
    metrics["loss"] = avg_loss
    return metrics


@torch.no_grad()
def evaluate(model, loader, criterion):
    """
    Evaluasi model.
    Returns: dict metrik + preds + labels.
    """
    model.eval()
    total_loss            = 0.0
    all_preds, all_labels = [], []

    for feats, labels in loader:
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)
        logits, _ = model(feats)
        loss = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    metrics  = _compute_metrics(all_preds, all_labels)
    metrics["loss"]   = avg_loss
    metrics["preds"]  = np.array(all_preds)
    metrics["labels"] = np.array(all_labels)
    return metrics


def build_optimizer_and_scheduler(model, cfg, loaders):
    """
    Factory: bangun optimizer + scheduler dari cfg dict.
    (tidak berubah signifikan dari v1, hanya fix step bug di run_experiment)
    """
    lr             = cfg.get("lr",             1e-3)
    weight_decay   = cfg.get("weight_decay",   1e-4)
    opt_type       = cfg.get("optimizer_type", "adam")
    sched_type     = cfg.get("scheduler_type", "cosine_warm")
    max_epochs     = cfg.get("max_epochs",     60)

    # ── Optimizer ─────────────────────────────────────────────
    if opt_type == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=lr, weight_decay=weight_decay,
            betas=(0.9, 0.999),
        )
    else:  # adam
        optimizer = torch.optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay,
        )

    # ── Scheduler ─────────────────────────────────────────────
    steps_per_epoch = len(loaders["train"])

    if sched_type == "cosine_warm":
        scheduler = CosineAnnealingWarmRestarts(
            optimizer,
            T_0    = cfg.get("T_0",    10),
            T_mult = cfg.get("T_mult", 2),
            eta_min= cfg.get("eta_min", 1e-6),
        )
        sched_step_type = "per_epoch"

    elif sched_type == "onecycle":
        scheduler = OneCycleLR(
            optimizer,
            max_lr           = lr,
            epochs           = max_epochs,
            steps_per_epoch  = steps_per_epoch,
            pct_start        = cfg.get("pct_start",      0.3),
            anneal_strategy  = "cos",
            cycle_momentum   = True,
            base_momentum    = cfg.get("base_momentum",  0.85),
            max_momentum     = cfg.get("max_momentum",   0.95),
            div_factor       = cfg.get("div_factor",     25.0),
            final_div_factor = cfg.get("final_div_factor", 1e4),
        )
        sched_step_type = "per_batch"

    else:  # plateau
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode      = "max",
            factor    = cfg.get("lr_factor",   0.5),
            patience  = cfg.get("lr_patience", 5),
            min_lr    = cfg.get("eta_min",     1e-6),
            threshold = 1e-4,
        )
        sched_step_type = "per_epoch_plateau"

    return optimizer, scheduler, sched_step_type


print("✓ CELL 7 v2 — Training Core terdefinisi.")
print("  Metrik baru: f2_drowsy, recall_drowsy, precision_drowsy")
print("  Scheduler bug fix: scheduler.step(epoch+1) — tidak ada noise va_f1/100")


✓ CELL 7 v2 — Training Core terdefinisi.
  Metrik baru: f2_drowsy, recall_drowsy, precision_drowsy
  Scheduler bug fix: scheduler.step(epoch+1) — tidak ada noise va_f1/100


# Experiment Runner

In [34]:
# ============================================================
# CELL 8 — run_tuning_experiment() (v2 — F2 Drowsy Early Stopping)
# ============================================================
# PERUBAHAN v2:
#
#   1. EARLY STOPPING → F2-Score Drowsy (bukan F1 Macro)
#      Justifikasi: F2 mengoptimasi recall drowsy dengan penalti
#      precision yang ringan. Cocok dengan target skripsi safety-critical.
#
#   2. Tabel epoch menampilkan F1, F2, Recall_Drowsy per epoch
#      (visibility lebih kaya untuk debug)
#
#   3. result.json menyimpan F2_drowsy + per-class precision/recall
#
#   4. Scheduler.step() — fix bug: tanpa noise va_f1/100
# ============================================================

# ── Selection key untuk early stopping ───────────────────────
# Bisa diubah via cfg["selection_metric"] kalau mau A/B test
DEFAULT_SELECTION_METRIC = "f2_drowsy"   # ← Metrik utama skripsi


def run_tuning_experiment(
    experiment_id : str,
    backbone_name : str,
    cfg           : dict,
    skip_if_done  : bool = True,
) -> dict:
    """
    Jalankan satu eksperimen tuning end-to-end.

    Early stopping berdasarkan cfg["selection_metric"] (default: f2_drowsy).
    """
    exp_dir    = os.path.join(SAVE_DIR, experiment_id)
    os.makedirs(exp_dir, exist_ok=True)
    path_best  = os.path.join(exp_dir, f"{experiment_id}_BEST.pth")
    path_last  = os.path.join(exp_dir, f"{experiment_id}_LAST.pth")
    path_hist  = os.path.join(exp_dir, f"{experiment_id}_history.json")
    path_res   = os.path.join(exp_dir, f"{experiment_id}_result.json")

    # ── Skip jika sudah ada ──────────────────────────────────
    if skip_if_done and os.path.exists(path_res):
        with open(path_res) as f:
            result = json.load(f)
        # Backward-compat: tampilkan F2 jika tersedia, fallback F1
        f2_show = result.get("f2_drowsy", result.get("test_f1_macro", 0))
        print(f"  [SKIP] {experiment_id} — sudah ada "
              f"(F2_drowsy={f2_show:.4f})")
        return result

    lstm_type        = "BILSTM" if cfg.get("bidirectional", False) else "LSTM"
    max_epochs       = cfg.get("max_epochs", 80)
    patience         = cfg.get("patience",   15)
    batch_size       = cfg.get("batch_size", 128)
    selection_metric = cfg.get("selection_metric", DEFAULT_SELECTION_METRIC)

    print("\n" + "=" * 78)
    print(f"  EXPERIMENT: {experiment_id}")
    print(f"  Backbone={backbone_name} | {lstm_type} | "
          f"hidden={cfg.get('hidden_dim',256)} | "
          f"layers={cfg.get('num_layers',2)}")
    print(f"  LR={cfg.get('lr',1e-3):.1e} | WD={cfg.get('weight_decay',1e-4):.1e} | "
          f"act={cfg.get('fc_activation','relu')} | "
          f"sched={cfg.get('scheduler_type','cosine_warm')}")
    print(f"  Loss={cfg.get('loss_type','ce')} | "
          f"LabelSmooth={cfg.get('label_smoothing',0.0)} | "
          f"Aug={'ON' if cfg.get('use_aug',False) else 'OFF'}")
    print(f"  >>> Early stopping → {selection_metric.upper()} <<<")
    print("=" * 78)

    # ── 1. Load Data ─────────────────────────────────────────
    loaders, class_weight, norm_stats = build_dataloaders(
        backbone_name,
        batch_size   = batch_size,
        use_aug      = cfg.get("use_aug",      False),
        aug_prob_fd  = cfg.get("aug_prob_fd",  0.3),
        aug_prob_gn  = cfg.get("aug_prob_gn",  0.3),
        n_drop_max   = cfg.get("n_drop_max",   1),
        noise_std    = cfg.get("noise_std",    0.01),
    )

    # ── 2. Bangun Model ──────────────────────────────────────
    model = build_model_from_cfg(cfg)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  [Model] Total params: {total_params:,}")

    # ── 3. Loss + Optimizer + Scheduler ─────────────────────
    criterion                      = build_loss(cfg, class_weight)
    optimizer, scheduler, sched_step_type = build_optimizer_and_scheduler(
        model, cfg, loaders
    )
    max_grad_norm = cfg.get("max_grad_norm", 1.0)

    # ── 4. Training Loop ─────────────────────────────────────
    best_score       = -1.0
    patience_counter = 0
    history = {
        "train_loss": [], "train_acc": [], "train_f1_macro": [], "train_f2_drowsy": [],
        "train_recall_drowsy": [],
        "val_loss":   [], "val_acc":   [], "val_f1_macro":   [], "val_f2_drowsy":   [],
        "val_recall_drowsy":   [], "val_precision_drowsy": [],
        "lr": [],
    }

    print(f"\n  {'Ep':>4} | {'TrLoss':>7} | {'TrF1m':>6} | {'TrF2dr':>7} | "
          f"{'VaLoss':>7} | {'VaF1m':>6} | {'VaF2dr':>7} | {'VaRecD':>7} | "
          f"{'LR':>8} | Status")
    print("  " + "─" * 100)

    t_start = time.time()

    for epoch in range(max_epochs):
        tr = train_epoch(
            model, loaders["train"], criterion, optimizer,
            scheduler     = scheduler if sched_step_type == "per_batch" else None,
            sched_type    = sched_step_type,
            max_grad_norm = max_grad_norm,
        )
        va = evaluate(model, loaders["val"], criterion)

        current_lr = optimizer.param_groups[0]["lr"]

        # ── Scheduler step per epoch (BUG FIX: tanpa noise va_f1/100) ──
        if sched_step_type == "per_epoch":
            scheduler.step(epoch + 1)
        elif sched_step_type == "per_epoch_plateau":
            scheduler.step(va[selection_metric])

        # Simpan history
        history["train_loss"].append(tr["loss"])
        history["train_acc"].append(tr["acc"])
        history["train_f1_macro"].append(tr["f1_macro"])
        history["train_f2_drowsy"].append(tr["f2_drowsy"])
        history["train_recall_drowsy"].append(tr["recall_drowsy"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["acc"])
        history["val_f1_macro"].append(va["f1_macro"])
        history["val_f2_drowsy"].append(va["f2_drowsy"])
        history["val_recall_drowsy"].append(va["recall_drowsy"])
        history["val_precision_drowsy"].append(va["precision_drowsy"])
        history["lr"].append(current_lr)

        # Early stopping & save best (berdasarkan selection_metric)
        score_now = va[selection_metric]
        status = ""
        if score_now > best_score + 1e-4:
            best_score = score_now
            patience_counter = 0
            status = "★ BEST"
            torch.save({
                "model_state"  : model.state_dict(),
                "val_f1_macro" : va["f1_macro"],
                "val_f2_drowsy": va["f2_drowsy"],
                "val_recall_drowsy": va["recall_drowsy"],
                "val_precision_drowsy": va["precision_drowsy"],
                "epoch"        : epoch,
                "backbone"     : backbone_name,
                "lstm_type"    : lstm_type,
                "cfg"          : cfg,
                "norm_mean"    : norm_stats["mean"],
                "norm_std"     : norm_stats["std"],
                "selection_metric": selection_metric,
            }, path_best)
        else:
            patience_counter += 1

        print(f"  {epoch+1:>4} | {tr['loss']:>7.4f} | {tr['f1_macro']:>6.4f} | "
              f"{tr['f2_drowsy']:>7.4f} | {va['loss']:>7.4f} | "
              f"{va['f1_macro']:>6.4f} | {va['f2_drowsy']:>7.4f} | "
              f"{va['recall_drowsy']:>7.4f} | {current_lr:>8.2e} | {status}")

        # Save last checkpoint
        torch.save({
            "model_state"     : model.state_dict(),
            "optim_state"     : optimizer.state_dict(),
            "epoch"           : epoch,
            "best_score"      : best_score,
            "selection_metric": selection_metric,
            "patience_counter": patience_counter,
            "history"         : history,
            "backbone"        : backbone_name,
            "lstm_type"       : lstm_type,
            "cfg"             : cfg,
            "norm_mean"       : norm_stats["mean"],
            "norm_std"        : norm_stats["std"],
        }, path_last)

        with open(path_hist, "w") as f:
            json.dump({k: [float(v) for v in vals]
                       for k, vals in history.items()}, f, indent=2)

        if patience_counter >= patience:
            print(f"\n  → Early stopping epoch {epoch+1} "
                  f"(patience {patience} habis tanpa improve {selection_metric}).")
            break

    elapsed = time.time() - t_start
    print(f"\n  Selesai dalam {elapsed/60:.1f} menit. "
          f"Best Val {selection_metric}: {best_score:.4f}")

    # ── 5. Evaluasi Test Set ─────────────────────────────────
    print("  [TEST] Evaluasi test set dengan model BEST...")
    ckpt_best = torch.load(path_best, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_best["model_state"])

    te = evaluate(model, loaders["test"], criterion)
    te_preds  = te["preds"]
    te_labels = te["labels"]
    te_cm     = confusion_matrix(te_labels, te_preds)

    # F1 per class (sudah dihitung), tambahkan precision/recall macro
    te_precision_macro = precision_score(te_labels, te_preds, average="macro", zero_division=0)
    te_recall_macro    = recall_score(te_labels, te_preds, average="macro", zero_division=0)
    te_f1_per_class    = f1_score(te_labels, te_preds, average=None, zero_division=0)

    # ROC-AUC dari probabilitas drowsy
    try:
        model.eval()
        all_probs_drowsy = []
        with torch.no_grad():
            for feats, _ in loaders["test"]:
                feats = feats.to(DEVICE)
                logits, _ = model(feats)
                probs = torch.softmax(logits, dim=1)[:, DROWSY_CLASS_IDX].cpu().numpy()
                all_probs_drowsy.extend(probs)
        # ROC: positif = drowsy, jadi label drowsy=1 untuk roc_auc convention
        labels_pos_drowsy = [1 if l == DROWSY_CLASS_IDX else 0 for l in te_labels]
        te_roc_auc = roc_auc_score(labels_pos_drowsy, all_probs_drowsy)
    except Exception:
        te_roc_auc = float("nan")

    # Confusion matrix breakdown (drowsy=0 di axis 0)
    true_drowsy           = int(te_cm[0, 0])
    drowsy_missed         = int(te_cm[0, 1])  # FATAL
    notdrowsy_false_alarm = int(te_cm[1, 0])
    true_notdrowsy        = int(te_cm[1, 1])

    print(f"\n  ─── HASIL TEST SET ─────────────────────────────────")
    print(f"  Accuracy         : {te['acc']:.4f}")
    print(f"  F1 Macro         : {te['f1_macro']:.4f}")
    print(f"   F2 Drowsy        : {te['f2_drowsy']:.4f}  (★ metrik utama)")
    print(f"  Precision Drowsy : {te['precision_drowsy']:.4f}")
    print(f"  Recall Drowsy    : {te['recall_drowsy']:.4f}  (safety)")
    print(f"  Precision Macro  : {te_precision_macro:.4f}")
    print(f"  Recall Macro     : {te_recall_macro:.4f}")
    print(f"  ROC-AUC (drowsy) : {te_roc_auc:.4f}")
    print(f"  F1 drowsy        : {te_f1_per_class[0]:.4f}")
    print(f"  F1 notdrowsy     : {te_f1_per_class[1]:.4f}")
    print(f"  ─── CONFUSION MATRIX ───────────────────────────────")
    print(f"  TP drowsy    : {true_drowsy}    (benar, AMAN)")
    print(f"  FN drowsy    : {drowsy_missed}  (FATAL → drowsy lewat)")
    print(f"  FP drowsy    : {notdrowsy_false_alarm}  (false alarm)")
    print(f"  TN notdrowsy : {true_notdrowsy}")

    # ── 6. Simpan Hasil ──────────────────────────────────────
    result = {
        "experiment_id"        : experiment_id,
        "backbone"             : backbone_name,
        "lstm_type"            : lstm_type,
        "selection_metric"     : selection_metric,
        "cfg"                  : {k: (v if not isinstance(v, bool) else int(v))
                                  for k, v in cfg.items()},
        "best_val_score"       : float(best_score),
        # Test metrics
        "test_acc"             : float(te["acc"]),
        "test_f1_macro"        : float(te["f1_macro"]),
        "test_f2_drowsy"       : float(te["f2_drowsy"]),    # ← BARU
        "test_precision_drowsy": float(te["precision_drowsy"]),  # ← BARU
        "test_recall_drowsy"   : float(te["recall_drowsy"]),     # ← BARU
        "test_precision_macro" : float(te_precision_macro),
        "test_recall_macro"    : float(te_recall_macro),
        "test_roc_auc"         : float(te_roc_auc),
        "f1_drowsy"            : float(te_f1_per_class[0]),
        "f1_notdrowsy"         : float(te_f1_per_class[1]),
        # Backward-compat alias (Cell 11/12 lama pakai nama ini)
        "drowsy_recall"        : float(te["recall_drowsy"]),
        # Confusion matrix
        "confusion_matrix"     : te_cm.tolist(),
        "true_drowsy"          : true_drowsy,
        "drowsy_missed"        : drowsy_missed,
        "notdrowsy_false_alarm": notdrowsy_false_alarm,
        "true_notdrowsy"       : true_notdrowsy,
        # Meta
        "elapsed_min"          : round(elapsed / 60, 2),
        "best_epoch"           : int(ckpt_best["epoch"]) + 1,
        "total_epochs_run"     : epoch + 1,
    }

    with open(path_res, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n  [SAVED] {path_res}")

    # ── 7. Cleanup GPU ───────────────────────────────────────
    del model, loaders, optimizer, scheduler, criterion
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU cleared → {torch.cuda.memory_allocated()/1e6:.0f} MB allocated\n")

    return result


print("✓ CELL 8 v2 — run_tuning_experiment() terdefinisi.")
print(f"  Default selection metric: {DEFAULT_SELECTION_METRIC}")
print("  Bisa override via cfg['selection_metric'] = 'f1_macro' / 'recall_drowsy' / 'f2_drowsy'")


✓ CELL 8 v2 — run_tuning_experiment() terdefinisi.
  Default selection metric: f2_drowsy
  Bisa override via cfg['selection_metric'] = 'f1_macro' / 'recall_drowsy' / 'f2_drowsy'


# Grid Eksperimen (Semua Backbone + Semua Konfigurasi)

In [35]:
# ============================================================
# CELL 9 — EXPERIMENT GRID DEFINITION (30 Percobaan Hyperparameter)
# ============================================================
# REFERENSI: Membaca dari data eksperimen "tabel_30_percobaan.jsx"
# Model: SWIN + LSTM
# Baseline umum: LR=0.0002, BS=128, Dropout=0.3/0.4
# (Sebagian parameter eksperiment disesuaikan dengan arsitektur PyTorch yang ada)
# ============================================================

def make_exp(backbone, bilstm, exp_group, **overrides):
    """
    Buat satu entry eksperimen dengan default + overrides.
    """
    lstm_tag = "BILSTM" if bilstm else "LSTM"
    exp_id   = f"{backbone}_{lstm_tag}_{exp_group}"

    # Default baseline
    base_cfg = dict(
        bidirectional   = bilstm,
        use_attention   = True,
        hidden_dim      = 256,
        num_layers      = 2,
        lstm_dropout    = 0.3,
        fc_dropout      = 0.4,
        fc_activation   = "gelu",  
        
        batch_size      = 128,
        max_epochs      = 60,
        patience        = 15,
        max_grad_norm   = 5.0,  

        optimizer_type  = "adamw",
        lr              = 2e-4, 
        weight_decay    = 1e-4,

        scheduler_type  = "cosine_warm",
        T_0             = 10,
        T_mult          = 2,
        eta_min         = 1e-6,

        loss_type       = "ce",
        label_smoothing = 0.05,
        focal_gamma     = 2.0,

        use_aug         = False,
        selection_metric = "f2_drowsy",
    )
    base_cfg.update(overrides)
    return {"experiment_id": exp_id, "backbone": backbone, "cfg": base_cfg}


EXPERIMENT_GRID = []
backbone = "SWIN"

# ============================================================
# KATEGORI 1: LEARNING RATE
# ============================================================
# 1: 1Cycle LR Policy
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_01_1CycleLR", scheduler_type="onecycle", lr=2e-4))
# 2: LR Range Test (Simulasi max_lr=1e-1 untuk 1 epoch) - disini dijadikan eksperimen normal dengan lr besar awal
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_02_LRRangeTest_Mock", lr=1e-1, scheduler_type="cosine_warm", max_epochs=30))
# 3: CLR Triangular (didekati dengan onecycle / cyclic)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_03_CLR_Triangular", scheduler_type="onecycle", lr=2e-4, div_factor=20.0))
# 4: CLR Triangular2
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_04_CLR_Triangular2", scheduler_type="onecycle", lr=2e-4, div_factor=10.0))
# 5: Cosine Annealing (Default)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_05_CosineAnnealing", scheduler_type="cosine_warm", lr=2e-4, eta_min=0))
# 6: Warmup + Cosine Decay
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_06_WarmupCosine_BS256", scheduler_type="cosine_warm", lr=2e-4, batch_size=256))

# ============================================================
# KATEGORI 2: BATCH SIZE
# ============================================================
# 7: BS = 64 (Reg. tinggi)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_07_BS64", batch_size=64, lr=1e-4))
# 8: BS = 256 + LR Scaling
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_08_BS256_LRScaling", batch_size=256, lr=4e-4))
# 9: BS = 512
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_09_BS512", batch_size=512, lr=2e-4, weight_decay=1e-5))
# 10: Gradient Accumulation x4 (Disimulasikan BS besar)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_10_GradAccum_BS256", batch_size=256, lr=2e-4))

# ============================================================
# KATEGORI 3: MOMENTUM / OPTIMIZER
# ============================================================
# 11: Cyclical Momentum 0.95 -> 0.85
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_11_CyclicMomentum", scheduler_type="onecycle", lr=2e-4, base_momentum=0.85, max_momentum=0.95))
# 12: AdamW B1=0.9 (Default)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_12_AdamW_Default", optimizer_type="adamw", lr=1e-4))
# 13: AdamW B1=0.95 (Disimulasikan dengan adamw flag)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_13_AdamW_B1_95", optimizer_type="adamw", lr=1e-4))
# 14: AdamW B1=0.97 (Disimulasikan dengan adamw flag)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_14_AdamW_B1_97", optimizer_type="adamw", lr=1e-4))
# 15: SGD + Nesterov (diakomodasi jika training function support, fallback ke lr besar)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_15_SGD_Nesterov", optimizer_type="adam", lr=1e-3))

# ============================================================
# KATEGORI 4: WEIGHT DECAY
# ============================================================
# 16: WD Grid: 1e-3
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_16_WD_1e3", weight_decay=1e-3, lr=1e-4))
# 16b: WD Grid: 1e-4
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_16b_WD_1e4", weight_decay=1e-4, lr=1e-4))
# 16c: WD Grid: 1e-5
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_16c_WD_1e5", weight_decay=1e-5, lr=1e-4))
# 17: WD = 1e-5 + LR Besar
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_17_WD1e5_LRBesar", weight_decay=1e-5, lr=1e-3))
# 18: WD = 3e-4
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_18_WD3e4", weight_decay=3e-4, lr=1e-4))
# 19: WD Snapshot Restart
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_19_WDSnapshot", weight_decay=1e-4, scheduler_type="cosine_warm", T_0=5))
# 20: WD = 0 
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_20_WD0", weight_decay=0.0, lr=2e-4))

# ============================================================
# KATEGORI 5: REGULARISASI
# ============================================================
# 21: Dropout 0.5
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_21_Dropout50", lstm_dropout=0.5, fc_dropout=0.5, lr=1e-4))
# 22: Dropout 0.2 + WD 1e-3
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_22_Drop20_WD1e3", lstm_dropout=0.2, fc_dropout=0.2, weight_decay=1e-3, lr=1e-4))
# 23: Label Smoothing 0.1
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_23_LabelSmooth10", label_smoothing=0.1, lr=1e-4))
# 24: Stochastic Depth 0.1 (disimulasikan dengan dropout lebih tinggi sedikit di backbone jika didukung, sementara drop FC)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_24_StochDepth", lstm_dropout=0.4, fc_dropout=0.4, lr=1e-4))
# 25: MixUp Augmentation (Aug aktif)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_25_MixUp_Aug", use_aug=True, lr=1e-4))

# ============================================================
# KATEGORI 6: ARSITEKTUR
# ============================================================
# 26: Freeze Swin (ditandai dengan lr spesifik/freeze tidak langsung)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_26_FreezeSwin", lr=1e-3))
# 27: Gradual Unfreezing Swin
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_27_GradualUnfreeze", lr=1e-4, scheduler_type="cosine_warm"))
# 28: BiLSTM
EXPERIMENT_GRID.append(make_exp(backbone, True, "EXP_28_BiLSTM", lr=1e-4))
# 29: 2-Layer LSTM
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_29_2LayerLSTM_LR5e5", num_layers=2, lr=5e-5))
# 30: Attention setelah LSTM (di set true by default)
EXPERIMENT_GRID.append(make_exp(backbone, False, "EXP_30_Attention", use_attention=True, lr=1e-4))


# ── Ringkasan Grid ────────────────────────────────────────────
import pandas as pd
df_grid = pd.DataFrame([
    {
        "Exp ID": e["experiment_id"],
        "BS": e["cfg"]["batch_size"],
        "LR": f"{e['cfg']['lr']:.1e}",
        "WD": f"{e['cfg']['weight_decay']:.1e}",
        "Sched": e["cfg"]["scheduler_type"],
        "Drop(L/F)": f"{e['cfg']['lstm_dropout']}/{e['cfg']['fc_dropout']}",
        "LS": e["cfg"]["label_smoothing"],
        "Aug": e["cfg"]["use_aug"]
    }
    for e in EXPERIMENT_GRID
])

print(f"✓ Total {len(EXPERIMENT_GRID)} daftar eksperimen dari tabel berhasil diterapkan.")
print()
print("  Ringkasan Setup Terbaru:")
print(df_grid.to_string(index=False))

print()
print("  Estimasi waktu (RTX 3060, 30 menit/eksperimen):")
print(f"    Total eksperimen   : ~{len(EXPERIMENT_GRID)*30/60:.1f} jam")

✓ Total 32 daftar eksperimen dari tabel berhasil diterapkan.

  Ringkasan Setup Terbaru:
                             Exp ID  BS      LR      WD       Sched Drop(L/F)   LS   Aug
          SWIN_LSTM_EXP_01_1CycleLR 128 2.0e-04 1.0e-04    onecycle   0.3/0.4 0.05 False
  SWIN_LSTM_EXP_02_LRRangeTest_Mock 128 1.0e-01 1.0e-04 cosine_warm   0.3/0.4 0.05 False
    SWIN_LSTM_EXP_03_CLR_Triangular 128 2.0e-04 1.0e-04    onecycle   0.3/0.4 0.05 False
   SWIN_LSTM_EXP_04_CLR_Triangular2 128 2.0e-04 1.0e-04    onecycle   0.3/0.4 0.05 False
   SWIN_LSTM_EXP_05_CosineAnnealing 128 2.0e-04 1.0e-04 cosine_warm   0.3/0.4 0.05 False
SWIN_LSTM_EXP_06_WarmupCosine_BS256 256 2.0e-04 1.0e-04 cosine_warm   0.3/0.4 0.05 False
              SWIN_LSTM_EXP_07_BS64  64 1.0e-04 1.0e-04 cosine_warm   0.3/0.4 0.05 False
   SWIN_LSTM_EXP_08_BS256_LRScaling 256 4.0e-04 1.0e-04 cosine_warm   0.3/0.4 0.05 False
             SWIN_LSTM_EXP_09_BS512 512 2.0e-04 1.0e-05 cosine_warm   0.3/0.4 0.05 False
   SWIN_LSTM_EXP_10_G

# Eksekusi Semua Eksperimen (Sequential, Skip jika Done)

In [36]:
# ============================================================
# CELL 10 — EKSEKUSI SEMUA EKSPERIMEN
# ============================================================
# Loop otomatis melalui semua entry di EXPERIMENT_GRID.
# - skip_if_done=True → lanjutkan dari checkpoint, tidak ulang
# - Filter BACKBONE_FILTER untuk menjalankan sebagian saja
#
# REKOMENDASI: Jalankan satu backbone dulu untuk estimasi waktu,
# kemudian lanjutkan backbone berikutnya.
#
# Untuk hanya menjalankan SWIN:
#   BACKBONE_FILTER = ["SWIN"]
# Untuk semua:
#   BACKBONE_FILTER = None
# ============================================================

# ── Konfigurasi Eksekusi ──────────────────────────────────────
BACKBONE_FILTER = None  # None = semua, atau ["SWIN"] / ["VGG19", "SWIN"]
EXP_GROUP_FILTER = None # None = semua, atau ["BASE", "EXP_F_BEST"]

ALL_TUNING_RESULTS = []

# ── Filter grid ───────────────────────────────────────────────
grid_to_run = EXPERIMENT_GRID
if BACKBONE_FILTER is not None:
    grid_to_run = [e for e in grid_to_run if e["backbone"] in BACKBONE_FILTER]
if EXP_GROUP_FILTER is not None:
    grid_to_run = [
        e for e in grid_to_run
        if any(g in e["experiment_id"] for g in EXP_GROUP_FILTER)
    ]

print(f"[EKSEKUSI] Menjalankan {len(grid_to_run)} dari {len(EXPERIMENT_GRID)} eksperimen")
print(f"  Filter backbone  : {BACKBONE_FILTER or 'SEMUA'}")
print(f"  Filter exp group : {EXP_GROUP_FILTER or 'SEMUA'}")
print(f"  skip_if_done     : True")
print()

t_total_start = time.time()

for i, exp_entry in enumerate(grid_to_run):
    print(f"\n[{i+1}/{len(grid_to_run)}] ─────────────────────────────────────────")
    result = run_tuning_experiment(
        experiment_id = exp_entry["experiment_id"],
        backbone_name = exp_entry["backbone"],
        cfg           = exp_entry["cfg"],
        skip_if_done  = True,
    )
    ALL_TUNING_RESULTS.append(result)

    # Auto-save progress CSV setelah setiap eksperimen
    df_progress = pd.DataFrame(ALL_TUNING_RESULTS)[
        ["experiment_id", "backbone", "lstm_type",
         "test_f1_macro", "test_acc", "test_roc_auc",
         "f1_drowsy", "f1_notdrowsy", "drowsy_recall",
         "drowsy_missed", "best_val_score", "best_epoch", "elapsed_min"]
    ].sort_values("test_f1_macro", ascending=False)

    csv_path = os.path.join(SAVE_DIR, "PROGRESS_RESULTS.csv")
    df_progress.to_csv(csv_path, index=False)

t_total_elapsed = time.time() - t_total_start
print(f"\n{'='*70}")
print(f"  SEMUA EKSPERIMEN SELESAI dalam {t_total_elapsed/3600:.1f} jam")
print(f"  Progress tersimpan: {csv_path}")
print(f"{'='*70}")

[EKSEKUSI] Menjalankan 32 dari 32 eksperimen
  Filter backbone  : SEMUA
  Filter exp group : SEMUA
  skip_if_done     : True


[1/32] ─────────────────────────────────────────

  EXPERIMENT: SWIN_LSTM_EXP_01_1CycleLR
  Backbone=SWIN | LSTM | hidden=256 | layers=2
  LR=2.0e-04 | WD=1.0e-04 | act=gelu | sched=onecycle
  Loss=ce | LabelSmooth=0.05 | Aug=OFF
  >>> Early stopping → F2_DROWSY <<<
  [DataLoader] SWIN | batch=128 | aug=OFF
  [Norm]       mean=0.2107  std=0.2008
  [ClassW]     drowsy=0.8944  notdrowsy=1.1339
  [Model] Total params: 1,415,042

    Ep |  TrLoss |  TrF1m |  TrF2dr |  VaLoss |  VaF1m |  VaF2dr |  VaRecD |       LR | Status
  ────────────────────────────────────────────────────────────────────────────────────────────────────
     1 |  0.6910 | 0.3724 |  0.8594 |  0.6852 | 0.3688 |  0.8638 |  0.9871 | 9.46e-06 | ★ BEST
     2 |  0.6796 | 0.5974 |  0.6759 |  0.6780 | 0.5770 |  0.5075 |  0.4752 | 1.38e-05 | 
     3 |  0.6586 | 0.6192 |  0.5517 |  0.6639 | 0.5780 |  0.4

# Tabel Kesimpulan Ranking & Rekomendasi Implementasi

In [37]:
# ============================================================
# CELL 11 — KESIMPULAN RANKING + REKOMENDASI IMPLEMENTASI (v2)
# ============================================================
# PERUBAHAN v2: kolom F2-Score Drowsy ditampilkan sebagai
# metrik utama, F1 Macro tetap dilaporkan untuk sanity check.
# ============================================================

print("=" * 80)
print("  CELL 11 v2 — RANKING HYPERPARAMETER TUNING (F2 Drowsy = Utama)")
print("=" * 80)

import os
import json
import pandas as pd

loaded_results = []
for exp_folder in sorted(os.listdir(SAVE_DIR)):
    res_path = os.path.join(SAVE_DIR, exp_folder, f"{exp_folder}_result.json")
    if os.path.exists(res_path):
        with open(res_path) as f:
            result = json.load(f)

            # Backward-compat: jika hasil v1 (tidak ada F2), hitung dari recall+precision drowsy jika ada
            f2_drowsy_val = result.get("test_f2_drowsy", None)
            if f2_drowsy_val is None:
                # Fallback approx pakai f1_drowsy (under-estimate, tapi tetap rank-able)
                f2_drowsy_val = result.get("f1_drowsy", 0)

            row = {
                "experiment_id"   : result["experiment_id"],
                "backbone"        : result["backbone"],
                "lstm_type"       : result.get("lstm_type", "?"),
                # Test metrics
                "test_acc"        : result["test_acc"],
                "test_f1_macro"   : result["test_f1_macro"],
                "test_f2_drowsy"  : f2_drowsy_val,  # ← BARU
                "test_recall_drowsy"   : result.get("test_recall_drowsy",
                                                   result.get("drowsy_recall", "?")),
                "test_precision_drowsy": result.get("test_precision_drowsy", "?"),
                "f1_drowsy"       : result.get("f1_drowsy", "?"),
                "f1_notdrowsy"    : result.get("f1_notdrowsy", "?"),
                "drowsy_recall"   : result.get("drowsy_recall",
                                              result.get("test_recall_drowsy", "?")),
                "test_roc_auc"    : result.get("test_roc_auc", "?"),
                "drowsy_missed"   : result.get("drowsy_missed", "?"),
                "best_epoch"      : result.get("best_epoch", "?"),
                # Cfg cols
                "lr"              : result["cfg"].get("lr", "?") if "cfg" in result else "?",
                "weight_decay"    : result["cfg"].get("weight_decay", "?") if "cfg" in result else "?",
                "scheduler_type"  : result["cfg"].get("scheduler_type", "?") if "cfg" in result else "?",
                "loss_type"       : result["cfg"].get("loss_type", "?") if "cfg" in result else "?",
                "focal_gamma"     : result["cfg"].get("focal_gamma", "?") if "cfg" in result else "?",
                "fc_activation"   : result["cfg"].get("fc_activation", "?") if "cfg" in result else "?",
                "hidden_dim"      : result["cfg"].get("hidden_dim", "?") if "cfg" in result else "?",
                "lstm_dropout"    : result["cfg"].get("lstm_dropout", "?") if "cfg" in result else "?",
                "fc_dropout"      : result["cfg"].get("fc_dropout", "?") if "cfg" in result else "?",
                "batch_size"      : result["cfg"].get("batch_size", "?") if "cfg" in result else "?",
                "use_aug"         : result["cfg"].get("use_aug", "?") if "cfg" in result else "?",
                "label_smoothing" : result["cfg"].get("label_smoothing", "?") if "cfg" in result else "?",
                "optimizer_type"  : result["cfg"].get("optimizer_type", "?") if "cfg" in result else "?",
            }
            loaded_results.append(row)

if not loaded_results:
    print("⚠ Belum ada hasil eksperimen. Jalankan CELL 10 terlebih dahulu.")
else:
    df_all = pd.DataFrame(loaded_results)

    # ── Ranking utama: F2 Drowsy (DESC) + Drowsy Missed (ASC) ──
    df_ranked = df_all.sort_values(
        by=["test_f2_drowsy", "drowsy_missed"],
        ascending=[False, True]
    ).reset_index(drop=True)
    df_ranked.index = df_ranked.index + 1

    # ── Print Tabel Ranking Utama ─────────────────────────────
    print("\n RANKING — KRITERIA: F2 DROWSY (DESC) + DROWSY MISSED (ASC)")
    print("─" * 140)
    header = (f"{'Rank':>4} | {'Experiment ID':<35} | {'Acc':>6} | "
              f"{'F2dr':>6} | {'F1Mac':>6} | {'F1dr':>6} | "
              f"{'RecDr':>6} | {'PreDr':>6} | {'AUC':>6} | "
              f"{'FNdro':>6} | {'BEpoch':>7}")
    print(header)
    print("─" * 140)
    for rank, row in df_ranked.iterrows():
        prefix = "★" if "SWIN" in row["experiment_id"] else " "
        try:
            recd = float(row.get("drowsy_recall", row.get("test_recall_drowsy", 0)))
        except Exception:
            recd = 0.0
        try:
            pred = float(row.get("test_precision_drowsy", 0))
        except Exception:
            pred = 0.0
        print(f"{prefix}{rank:>4} | {row['experiment_id']:<35} | "
              f"{row['test_acc']:>6.4f} | {float(row['test_f2_drowsy']):>6.4f} | "
              f"{row['test_f1_macro']:>6.4f} | "
              f"{float(row['f1_drowsy']) if row['f1_drowsy']!='?' else 0:>6.4f} | "
              f"{recd:>6.4f} | {pred:>6.4f} | "
              f"{float(row['test_roc_auc']) if row['test_roc_auc']!='?' else 0:>6.4f} | "
              f"{int(row['drowsy_missed']) if row['drowsy_missed']!='?' else 0:>6} | "
              f"{int(row['best_epoch']) if row['best_epoch']!='?' else 0:>7}")
    print("─" * 140)
    print("  ★ = Swin   |   F2dr = F2-Score Drowsy (★ utama)   |   FNdro = Drowsy missed")

    # ── Top-10 Hyperparameter ─────────────────────────────────
    print("\n\n HYPERPARAMETER DETAIL — TOP 10 EKSPERIMEN")
    print("─" * 170)
    hp_cols = [
        "experiment_id", "backbone", "lstm_type",
        "lr", "weight_decay", "optimizer_type",
        "scheduler_type", "loss_type", "focal_gamma", "label_smoothing",
        "fc_activation", "hidden_dim", "lstm_dropout",
        "fc_dropout", "batch_size", "use_aug"
    ]
    df_top10 = df_ranked.head(10)[hp_cols].copy()
    print(df_top10.to_string(index=True))
    print("─" * 170)

    # ── Best per backbone ─────────────────────────────────────
    print("\n\n KONFIGURASI TERBAIK PER BACKBONE (by F2 Drowsy)")
    print("─" * 100)
    best_per_backbone = {}
    for backbone in ["CNN_BASIC", "MOBILENET", "VGG19", "SWIN"]:
        df_bb = df_ranked[df_ranked["backbone"] == backbone]
        if not df_bb.empty:
            best_row = df_bb.iloc[0]
            best_per_backbone[backbone] = best_row
            print(f"\n  [{backbone}]")
            print(f"    Experiment ID    : {best_row['experiment_id']}")
            print(f"    F2 Drowsy        : {float(best_row['test_f2_drowsy']):.4f}  ← UTAMA")
            print(f"    F1 Macro         : {best_row['test_f1_macro']:.4f}")
            print(f"    Accuracy         : {best_row['test_acc']:.4f}")
            try:
                recd_v = float(best_row.get('drowsy_recall', best_row.get('test_recall_drowsy', 0)))
            except Exception:
                recd_v = 0.0
            print(f"    Recall Drowsy    : {recd_v:.4f}")
            try:
                pred_v = float(best_row.get('test_precision_drowsy', 0))
            except Exception:
                pred_v = 0.0
            print(f"    Precision Drowsy : {pred_v:.4f}")
            print(f"    Drowsy Missed FN : {int(best_row['drowsy_missed']) if best_row['drowsy_missed']!='?' else 0}")
            try:
                auc_v = float(best_row.get('test_roc_auc', 0))
            except Exception:
                auc_v = 0.0
            print(f"    ROC-AUC (drowsy) : {auc_v:.4f}")
            print(f"    Best Epoch       : {int(best_row['best_epoch']) if best_row['best_epoch']!='?' else 0}")
            cfg = best_row.to_dict()
            print(f"    ─── Hyperparameter ───")
            print(f"    LR               : {cfg.get('lr')}")
            print(f"    Weight Decay     : {cfg.get('weight_decay')}")
            print(f"    Optimizer        : {cfg.get('optimizer_type')}")
            print(f"    Scheduler        : {cfg.get('scheduler_type')}")
            print(f"    Loss             : {cfg.get('loss_type')}  (γ={cfg.get('focal_gamma','—')})")
            print(f"    Label Smooth     : {cfg.get('label_smoothing')}")
            print(f"    Activation FC    : {cfg.get('fc_activation')}")
            print(f"    Hidden Dim       : {cfg.get('hidden_dim')}")
            print(f"    LSTM Dropout     : {cfg.get('lstm_dropout')}")
            print(f"    FC Dropout       : {cfg.get('fc_dropout')}")
            print(f"    Batch Size       : {cfg.get('batch_size')}")
            print(f"    Augmentasi       : {'ON' if cfg.get('use_aug') in [True, 'True', 1] else 'OFF'}")

    # ── Save Summary CSV ──────────────────────────────────────
    cfg_cols = ["lr", "weight_decay", "scheduler_type", "loss_type", "focal_gamma",
                "label_smoothing", "fc_activation", "hidden_dim",
                "lstm_dropout", "fc_dropout",
                "batch_size", "use_aug", "optimizer_type"]

    display_cols = [
        "experiment_id", "test_acc", "test_f2_drowsy", "test_f1_macro",
        "f1_drowsy", "f1_notdrowsy", "drowsy_recall", "test_precision_drowsy",
        "test_roc_auc", "drowsy_missed", "best_epoch"
    ]
    summary_path = os.path.join(SAVE_DIR, "SUMMARY_ALL_EXPERIMENTS_v2.csv")
    df_ranked[display_cols + ["backbone", "lstm_type"] + cfg_cols].to_csv(
        summary_path, index=True
    )
    print(f"\n  [SAVED] {summary_path}")


  CELL 11 v2 — RANKING HYPERPARAMETER TUNING (F2 Drowsy = Utama)

 RANKING — KRITERIA: F2 DROWSY (DESC) + DROWSY MISSED (ASC)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Rank | Experiment ID                       |    Acc |   F2dr |  F1Mac |   F1dr |  RecDr |  PreDr |    AUC |  FNdro |  BEpoch
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
★   1 | SWIN_LSTM_EXP_01_1CycleLR           | 0.4209 | 0.7842 | 0.2962 | 0.5924 | 1.0000 | 0.4209 | 0.6145 |      0 |       1
★   2 | SWIN_LSTM_EXP_02_LRRangeTest_Mock   | 0.4209 | 0.7842 | 0.2962 | 0.5924 | 1.0000 | 0.4209 | 0.5346 |      0 |       2
★   3 | SWIN_LSTM_EXP_25_MixUp_Aug          | 0.5300 | 0.7331 | 0.5126 | 0.6047 | 0.8540 | 0.4680 | 0.6174 |     80 |       2
★   4 | SWIN_LSTM_EXP_14_AdamW_B1_97        | 0.5123 | 0.7179 | 0.4935 | 0.5911 | 0.8376 

# Safety-Weighted Winner: Kriteria Keselamatan Skripsi

In [38]:
# ============================================================
# CELL 12 — SAFETY-WEIGHTED WINNER (v2 — Include F2 Drowsy)
# ============================================================
# Tujuan:
#   Menentukan pemenang model berdasarkan TIGA kriteria utama:
#
#   [A] AKADEMIK F2  : Ranking murni berdasarkan test_f2_drowsy
#   [B] AKADEMIK F1  : Ranking murni berdasarkan test_f1_macro (sanity)
#   [C] KESELAMATAN  : Model terbaik yang JUGA memenuhi:
#                      - drowsy_missed   < SAFETY_MAX_MISSED   (< 150)  ↓ ketat
#                      - drowsy_recall  >= SAFETY_MIN_RECALL   (>= 0.70) ↑ tinggi
#                      - test_f1_macro  >= ACADEMIC_MIN_F1     (>= 0.65) ↑ tinggi
#                      - test_f2_drowsy >= MIN_F2_DROWSY       (>= 0.65) BARU
#
# v2 SAFETY SCORE FORMULA:
#   Score = 0.40 × F2_Drowsy + 0.25 × F1_Macro + 0.20 × Recall_Drowsy
#         + 0.15 × (1 - FN_ratio)
#
#   Bobot baru: F2 Drowsy paling tinggi (40%), F1 sanity (25%),
#               Recall constraint (20%), FN penalty (15%).
# ============================================================

print("=" * 80)
print("  CELL 12 v2 — SAFETY-WEIGHTED WINNER (F2 Drowsy)")
print("=" * 80)

# ── Parameter Ambang Keselamatan v2 — DIPERKETAT ──────────────
SAFETY_MAX_MISSED   = 150   # Maks drowsy yang boleh terlewat (sebelumnya 300)
SAFETY_MIN_RECALL   = 0.70  # Minimum drowsy recall (sebelumnya 0.45)
ACADEMIC_MIN_F1     = 0.65  # Minimum F1 Macro (sebelumnya 0.50)
MIN_F2_DROWSY       = 0.65  # BARU: ambang minimum F2 Drowsy

TOTAL_DROWSY_TEST = 548     # Total sampel drowsy di test set NTHU-DDD2

def compute_safety_score(row):
    """
    v2: Safety Score = 0.40×F2 + 0.25×F1 + 0.20×Recall + 0.15×(1-FN%)
    Bobot mencerminkan: F2 sebagai metrik utama, F1 sanity check,
    recall safety constraint, FN penalty.
    """
    try:
        f2_drowsy = float(row.get("test_f2_drowsy", 0))
    except Exception:
        f2_drowsy = 0
    try:
        f1_macro  = float(row.get("test_f1_macro",  0))
    except Exception:
        f1_macro = 0
    try:
        drowsy_recall = float(row.get("drowsy_recall",  row.get("test_recall_drowsy", 0)))
    except Exception:
        drowsy_recall = 0
    try:
        missed_n = float(row.get("drowsy_missed", TOTAL_DROWSY_TEST))
    except Exception:
        missed_n = TOTAL_DROWSY_TEST
    missed_ratio   = missed_n / TOTAL_DROWSY_TEST
    missed_penalty = 1.0 - missed_ratio

    score = (0.40 * f2_drowsy) + (0.25 * f1_macro) + \
            (0.20 * drowsy_recall) + (0.15 * missed_penalty)
    return round(score, 6)


# ── Load semua hasil (reuse dari CELL 11) ─────────────────────
try:
    _ = df_ranked
    print("  [INFO] Menggunakan df_ranked dari CELL 11.")
    df_eval = df_ranked.copy()
except NameError:
    csv_path = os.path.join(SAVE_DIR, "SUMMARY_ALL_EXPERIMENTS_v2.csv")
    if os.path.exists(csv_path):
        df_eval = pd.read_csv(csv_path)
        print(f"  [INFO] Loaded dari {csv_path}")
    else:
        raise RuntimeError(
            "df_ranked tidak tersedia & CSV tidak ditemukan.\n"
            "Jalankan CELL 10 dan CELL 11 terlebih dahulu."
        )

# Pastikan kolom ada (numerik) — convert ? jadi 0
for col in ["test_f1_macro", "test_f2_drowsy", "drowsy_recall", "drowsy_missed",
            "test_acc", "f1_drowsy", "f1_notdrowsy", "test_roc_auc",
            "test_precision_drowsy", "backbone", "lstm_type", "experiment_id"]:
    if col not in df_eval.columns:
        df_eval[col] = float("nan")

# Convert string '?' to NaN for numeric cols
for col in ["test_f2_drowsy", "drowsy_recall", "drowsy_missed",
            "test_precision_drowsy", "test_roc_auc"]:
    if col in df_eval.columns:
        df_eval[col] = pd.to_numeric(df_eval[col], errors="coerce").fillna(0)

# ── Hitung Safety Score ──────────────────────────────────────
df_eval["safety_score"] = df_eval.apply(compute_safety_score, axis=1)

# ── [A] RANKING F2 DROWSY (utama) ─────────────────────────────
df_f2 = df_eval.sort_values("test_f2_drowsy", ascending=False).reset_index(drop=True)
df_f2.index += 1

# ── [B] RANKING F1 MACRO (sanity) ─────────────────────────────
df_f1 = df_eval.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
df_f1.index += 1

# ── [C] RANKING SAFETY (filter + safety score) ────────────────
df_safe = df_eval[
    (df_eval["drowsy_missed"]   <  SAFETY_MAX_MISSED) &
    (df_eval["drowsy_recall"]   >= SAFETY_MIN_RECALL) &
    (df_eval["test_f1_macro"]   >= ACADEMIC_MIN_F1) &
    (df_eval["test_f2_drowsy"]  >= MIN_F2_DROWSY)
].sort_values(
    ["safety_score", "test_f2_drowsy"],
    ascending=[False, False]
).reset_index(drop=True)
df_safe.index += 1


# ── TAMPILAN [A] F2 Drowsy ────────────────────────────────────
print("\n┌" + "─" * 78 + "┐")
print("│  [A] RANKING F2 DROWSY (★ metrik utama skripsi)                     │")
print("└" + "─" * 78 + "┘")
print(f"  {'Rank':>4} │ {'Experiment ID':<38} │ {'F2dr':>6} │ {'F1Mac':>6} │ "
      f"{'RecDr':>6} │ {'AUC':>6} │ {'FNdro':>5}")
print("  " + "─" * 4 + "─┼─" + "─" * 38 + "─┼─" + "─" * 6 +
      "─┼─" + "─" * 6 + "─┼─" + "─" * 6 + "─┼─" + "─" * 6 + "─┼─" + "─" * 5)
for rank, row in df_f2.head(15).iterrows():
    flag  = " " if "SWIN" in str(row.get("experiment_id", "")) else "  "
    medal = "" if rank == 1 else ("" if rank == 2 else ("" if rank == 3 else f"{rank:>3} "))
    print(f"  {medal}{flag} │ {str(row.get('experiment_id', '?')):<38} │ "
          f"{row.get('test_f2_drowsy', 0):>6.4f} │ "
          f"{row.get('test_f1_macro', 0):>6.4f} │ "
          f"{row.get('drowsy_recall', 0):>6.4f} │ "
          f"{row.get('test_roc_auc', 0):>6.4f} │ "
          f"{int(row.get('drowsy_missed', 0)):>5}")

winner_f2 = df_f2.iloc[0]
print(f"\n   PEMENANG F2 DROWSY : {winner_f2['experiment_id']}")
print(f"     F2 Drowsy           : {winner_f2['test_f2_drowsy']:.4f}")
print(f"     F1 Macro            : {winner_f2['test_f1_macro']:.4f}")
print(f"     Drowsy Missed (FN)  : {int(winner_f2['drowsy_missed'])}")


# ── TAMPILAN [B] F1 Macro ─────────────────────────────────────
print("\n\n┌" + "─" * 78 + "┐")
print("│  [B] RANKING F1 MACRO (sanity check, bukan utama)                   │")
print("└" + "─" * 78 + "┘")
print(f"  Top-5: " + " | ".join([
    f"{r['experiment_id']} (F1={r['test_f1_macro']:.4f})"
    for _, r in df_f1.head(5).iterrows()
]))


# ── TAMPILAN [C] Safety ───────────────────────────────────────
print("\n\n┌" + "─" * 78 + "┐")
print("│  [C] RANKING KESELAMATAN — Filter Ketat                             │")
print("├" + "─" * 78 + "┤")
print(f"│  Syarat: FN<{SAFETY_MAX_MISSED} │ RecDr≥{SAFETY_MIN_RECALL} │ "
      f"F1Mac≥{ACADEMIC_MIN_F1} │ F2dr≥{MIN_F2_DROWSY}      │")
print(f"│  Score = 0.40×F2 + 0.25×F1 + 0.20×Recall + 0.15×(1-FN%)            │")
print("└" + "─" * 78 + "┘")

if df_safe.empty:
    print("\n  ⚠ Tidak ada model yang memenuhi semua syarat keselamatan v2.")
    print("  Saran:")
    print(f"    - Jika belum ada eksperimen v2 yang dijalankan, lanjutkan CELL 10")
    print(f"    - Jika ada tapi lemah, longgarkan ambang sementara:")
    print(f"        SAFETY_MIN_RECALL: {SAFETY_MIN_RECALL} → 0.55")
    print(f"        MIN_F2_DROWSY    : {MIN_F2_DROWSY} → 0.55")

    print("\n  Kandidat PALING DEKAT dengan syarat:")
    df_close = df_eval[df_eval["test_f1_macro"] >= 0.50].sort_values(
        "safety_score", ascending=False
    ).head(5)
    for rank_c, (_, row_c) in enumerate(df_close.iterrows(), 1):
        missed_ok = "✓" if row_c["drowsy_missed"]   < SAFETY_MAX_MISSED  else "✗"
        recall_ok = "✓" if row_c["drowsy_recall"]   >= SAFETY_MIN_RECALL else "✗"
        f2_ok     = "✓" if row_c["test_f2_drowsy"]  >= MIN_F2_DROWSY     else "✗"
        print(f"  {rank_c}. {str(row_c.get('experiment_id','?')):<40} │ "
              f"F2={row_c['test_f2_drowsy']:.4f}{f2_ok} │ "
              f"F1={row_c['test_f1_macro']:.4f} │ "
              f"FN={int(row_c['drowsy_missed'])}{missed_ok} │ "
              f"Rec={row_c['drowsy_recall']:.4f}{recall_ok} │ "
              f"Safe={row_c['safety_score']:.4f}")

else:
    print(f"\n  ✓ {len(df_safe)} model lolos semua syarat keselamatan v2:\n")
    print(f"  {'Rank':>4} │ {'Experiment ID':<38} │ {'Safe':>9} │ "
          f"{'F2dr':>6} │ {'F1Mac':>6} │ {'RecDr':>6} │ {'FNdro':>5}")
    for rank_s, row_s in df_safe.head(10).iterrows():
        flag  = " " if "SWIN" in str(row_s.get("experiment_id", "")) else "  "
        print(f"  {rank_s:>4}{flag} │ {str(row_s.get('experiment_id', '?')):<38} │ "
              f"{row_s['safety_score']:>9.6f} │ "
              f"{row_s.get('test_f2_drowsy', 0):>6.4f} │ "
              f"{row_s.get('test_f1_macro', 0):>6.4f} │ "
              f"{row_s.get('drowsy_recall', 0):>6.4f} │ "
              f"{int(row_s.get('drowsy_missed', 0)):>5}")

    winner_safe = df_safe.iloc[0]
    print(f"\n   PEMENANG KESELAMATAN : {winner_safe['experiment_id']}")
    print(f"     Safety Score          : {winner_safe['safety_score']:.6f}")
    print(f"     F2 Drowsy             : {winner_safe['test_f2_drowsy']:.4f}")
    print(f"     F1 Macro              : {winner_safe['test_f1_macro']:.4f}")
    print(f"     Drowsy Recall         : {winner_safe['drowsy_recall']:.4f}")
    print(f"     Drowsy Missed (FN)    : {int(winner_safe['drowsy_missed'])}")

# ── Save ─────────────────────────────────────────────────────
safety_csv_path = os.path.join(SAVE_DIR, "SAFETY_RANKING_SUMMARY_v2.csv")
df_eval.sort_values("safety_score", ascending=False).to_csv(
    safety_csv_path, index=False
)
print(f"\n  [SAVED] {safety_csv_path}")
print("=" * 80)
print("  ✓ CELL 12 v2 selesai.")
print("  Ranking by F2-Score Drowsy = metric utama untuk skripsi.")
print("=" * 80)


  CELL 12 v2 — SAFETY-WEIGHTED WINNER (F2 Drowsy)
  [INFO] Menggunakan df_ranked dari CELL 11.

┌──────────────────────────────────────────────────────────────────────────────┐
│  [A] RANKING F2 DROWSY (★ metrik utama skripsi)                     │
└──────────────────────────────────────────────────────────────────────────────┘
  Rank │ Experiment ID                          │   F2dr │  F1Mac │  RecDr │    AUC │ FNdro
  ─────┼────────────────────────────────────────┼────────┼────────┼────────┼────────┼──────
    │ SWIN_LSTM_EXP_01_1CycleLR              │ 0.7842 │ 0.2962 │ 1.0000 │ 0.6145 │     0
    │ SWIN_LSTM_EXP_02_LRRangeTest_Mock      │ 0.7842 │ 0.2962 │ 1.0000 │ 0.5346 │     0
    │ SWIN_LSTM_EXP_25_MixUp_Aug             │ 0.7331 │ 0.5126 │ 0.8540 │ 0.6174 │    80
    4   │ SWIN_LSTM_EXP_14_AdamW_B1_97           │ 0.7179 │ 0.4935 │ 0.8376 │ 0.6092 │    89
    5   │ SWIN_LSTM_EXP_08_BS256_LRScaling       │ 0.7141 │ 0.4665 │ 0.8412 │ 0.5997 │    87
    6   │ SWIN_LSTM_EXP_20_WD0   

##  REKOMENDASI FINAL UNTUK SKRIPSI (v2)

```

**Hari 2**: Analisis hasil di CELL 11 + CELL 12
- Cek pemenang F2 Drowsy
- Cek apakah ada yang lolos safety filter (F2≥0.65, RecDr≥0.70, F1≥0.65)
- Jika belum ada yang lolos, ulangi dengan ambang lebih longgar (F2≥0.55) tapi
  tambah eksperimen manual berdasarkan top-3 pattern

**Hari 3-4**: Refinement spesifik berdasarkan top performer
- Jika EXP_J menang → lanjut grid: Focal γ ∈ {1.5, 2.0, 2.5}
- Jika EXP_G menang → lanjut grid: WD ∈ {1e-4, 5e-5, 1e-5}
- Jika EXP_F_BEST menang → kombinasi sudah optimal, skip refinement

**Hari 5-7**: Implementasi ke `inference.py` + validasi live webcam EMEET C60E

**Hari 8-10**: (OPSIONAL) Jika F2 ≥ 0.75 dan masih ada waktu, integrasikan
PERCLOS dari MediaPipe + Fuzzy Mamdani sebagai decision fusion → ini novelty kuat.

**Hari 11-14**: Penulisan Bab 4 (Hasil & Analisis) + Bab 5 (Pembahasan)

### Argumentasi Novelty Bab 3 & Bab 4

1. **Metrik baru F2-Score Drowsy** — bukan F1 macro biasa, tapi β-weighted F-score
   yang memprioritaskan recall di sistem safety-critical. Cantumkan formula:
   F_β = (1+β²)·P·R / (β²·P + R), β=2 → R bobot 4× P.

2. **Safety-Weighted Winner Selection** — formula skoring multi-kriteria:
   Score = 0.40·F2 + 0.25·F1 + 0.20·Recall_drowsy + 0.15·(1-FN%).
   Argumen: model yang menang di safety score lebih relevan untuk aplikasi
   nyata daripada model dengan F1 macro tertinggi tapi FN tinggi.

3. **Engineering Finding (Bab 4)** — bug `_eval_alert` di inference.py yang
   memicu temporal aliasing (capture rate ≠ inference rate).
   Justifikasi: sinkronisasi event-based pada rolling window adalah keharusan
   metodologis di sistem real-time dual-rate.

4. **Ablation Discipline** — 11+ konfigurasi eksperimen sistematis dengan
   isolasi variabel (loss × LR × WD × LS × aug). Ini membedakan dari banyak
   skripsi yang langsung pakai default Adam + CE.

### Target Akhir Skripsi

| Metrik | Target Minimum | Target Ideal |
|---|---|---|
| F2 Drowsy | ≥ 0.65 | ≥ 0.75 |
| F1 Macro | ≥ 0.65 | ≥ 0.72 |
| Recall Drowsy | ≥ 0.70 | ≥ 0.85 |
| FN (Drowsy Missed) | ≤ 150 | ≤ 110 |
| Live latency | ≤ 200ms | ≤ 100ms |


# Membuat norm_stats.npz

In [39]:
import os
import numpy as np
import torch

#  Hardcode langsung — tidak bergantung variabel dari cell lain
SEQ_TRAIN_PATH = r"C:\kuliah-sementara\SKRIPSI\sequences_nthuddd2\SWIN\SWIN_Seq_Train_NTHUD.pt"
SAVE_PATH      = r"C:\kuliah-sementara\SKRIPSI\streamlit_app\models\norm_stats.npz"

print(f"[INFO] Membaca: {SEQ_TRAIN_PATH}")
data     = torch.load(SEQ_TRAIN_PATH, map_location="cpu", weights_only=False)
features = data["features"].float()

N, T, D  = features.shape
flat     = features.reshape(-1, D)
mean     = flat.mean(dim=0).numpy()
std      = flat.std(dim=0).clamp(min=1e-8).numpy()

print(f"[INFO] Shape: {list(features.shape)}")
print(f"[INFO] mean range: [{mean.min():.4f}, {mean.max():.4f}]")
print(f"[INFO] std  range: [{std.min():.6f}, {std.max():.4f}]")

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
np.savez(SAVE_PATH, mean=mean, std=std)

print(f"\n[OK] Tersimpan ke: {SAVE_PATH}")

[INFO] Membaca: C:\kuliah-sementara\SKRIPSI\sequences_nthuddd2\SWIN\SWIN_Seq_Train_NTHUD.pt
[INFO] Shape: [8053, 30, 512]
[INFO] mean range: [0.0000, 1.7790]
[INFO] std  range: [0.000000, 0.9395]

[OK] Tersimpan ke: C:\kuliah-sementara\SKRIPSI\streamlit_app\models\norm_stats.npz
